# 🏫 桃園國中 24 班智慧排課引擎 (Google Colab 一鍵執行版)

專為臺灣國中教務處教學組長設計：
- 支援 **全校 24 班 (701~908)** 自動最佳化排課
- 內建 **特教資源班 2~3 班同調抽離 (Math/Eng/Chinese 同步)**
- 內建 **週三本土語 (閩/客) 局部跑班鎖定**
- 內建 **體育課夏日第 5 節防呆避開**
- 求解完成後自動匯出 `timetable_data.json`，可直接套入您的日系可愛風、簡約黑白風、莫蘭迪風格網頁！

In [ ]:
# 步驟 1：安裝 Google 核心求解模組 (約 5 秒)
!pip install -q ortools openpyxl pandas

In [ ]:
# 步驟 2：載入演算法核心模組
import json
from ortools.sat.python import cp_model
from google.colab import files

print('✅ 環境準備完成！請在下一步上傳您的排課設定檔，或直接點選下一步使用內建 24 班示範資料。')

In [ ]:
# 步驟 3：產生示範設定資料 (包含 701~908 24班、週三本土語、特教抽離群組)
demo_config = {
    'classes': [f'{g}0{b}' for g in [7, 8, 9] for b in range(1, 9)],
    'teachers': {
        '陳國文': {'subject': '國文', 'role': '導師'},
        '林數學': {'subject': '數學', 'role': '專任'},
        '許數學': {'subject': '數學', 'role': '專任'},
        '王英語': {'subject': '英語', 'role': '專任'},
        '黃英文': {'subject': '英語', 'role': '導師'},
        '張自然': {'subject': '自然', 'role': '專任'},
        '李體育': {'subject': '體育', 'role': '專任'},
        '何綜合': {'subject': '綜合', 'role': '專任'},
        '吳社會': {'subject': '社會', 'role': '專任'},
        '趙藝術': {'subject': '藝術', 'role': '專任'},
        '劉科技': {'subject': '科技', 'role': '專任'}
    },
    'sync_groups': [
        {'subject': '數學', 'classes': ['701', '702']}, # 701 與 702 同時抽離數學到資源班
        {'subject': '數學', 'classes': ['703', '704']}, # 703 與 704 同時抽離數學到資源班
        {'subject': '英語', 'classes': ['801', '802']}  # 801 與 802 同時抽離英語到資源班
    ],
    'wed_groups': [
        {'subject': '本土語', 'classes': ['701', '702']},
        {'subject': '本土語', 'classes': ['703', '704']}
    ],
    'assignments': []
}

# 自動填充 24 班每週 35 節課
subjects_template = [
    ('國文', 5, '陳國文'),
    ('數學', 4, '林數學'),
    ('英語', 4, '王英語'),
    ('生物', 3, '張自然'),
    ('歷史', 1, '吳社會'),
    ('地理', 1, '吳社會'),
    ('公民', 1, '吳社會'),
    ('音樂', 1, '趙藝術'),
    ('視覺', 1, '趙藝術'),
    ('表藝', 1, '趙藝術'),
    ('體育', 3, '李體育'),
    ('童軍', 1, '何綜合'),
    ('家政', 1, '何綜合'),
    ('輔導', 1, '何綜合'),
    ('資訊', 1, '劉科技'),
    ('科技', 1, '劉科技'),
    ('本土語', 1, '本土語師'),
    ('班活', 1, '陳國文'),
    ('社團', 1, '社團師'),
    ('自習', 1, '陳國文'),
    ('彈性', 1, '陳國文')
]

for c in demo_config['classes']:
    for subj, per, t in subjects_template:
        demo_config['assignments'].append({
            'class_id': c,
            'subject': subj,
            'teacher': t if c == '701' else f'{t}_{c[0]}',
            'periods': per
        })

print(f'📊 示範配課資料建立完成，全校共 {len(demo_config["assignments"])} 筆課務！')

In [ ]:
# 步驟 4：啟動 Google CP-SAT 求解運算 (通常約 15~30 秒)
model = cp_model.CpModel()
DAYS = [1, 2, 3, 4, 5]
PERIODS = [1, 2, 3, 4, 5, 6, 7]
CLASSES = demo_config['classes']
ASSIGNMENTS = demo_config['assignments']
SYNC_GROUPS = demo_config['sync_groups']
WED_GROUPS = demo_config['wed_groups']

x = {}
for a_idx, a in enumerate(ASSIGNMENTS):
    for d in DAYS:
        for p in PERIODS:
            x[a_idx, d, p] = model.NewBoolVar(f'x_{a_idx}_{d}_{p}')

# 節數守恆
for a_idx, a in enumerate(ASSIGNMENTS):
    model.Add(sum(x[a_idx, d, p] for d in DAYS for p in PERIODS) == a['periods'])

# 班級時段互斥
for c in CLASSES:
    for d in DAYS:
        for p in PERIODS:
            c_assigns = [a_idx for a_idx, a in enumerate(ASSIGNMENTS) if a['class_id'] == c]
            model.Add(sum(x[a_idx, d, p] for a_idx in c_assigns) <= 1)

# 特教資源班 2~3 班同調抽離 (核心關鍵)
for group in SYNC_GROUPS:
    subj = group['subject']
    cls_list = group['classes']
    target_assigns = [a_idx for a_idx, a in enumerate(ASSIGNMENTS) if a['subject'] == subj and a['class_id'] in cls_list]
    if len(target_assigns) >= 2:
        first = target_assigns[0]
        for other in target_assigns[1:]:
            for d in DAYS:
                for p in PERIODS:
                    model.Add(x[first, d, p] == x[other, d, p])

# 週三本土語局部跑班 (強制排在週三 d=3)
for w_group in WED_GROUPS:
    subj = w_group['subject']
    cls_list = w_group['classes']
    w_assigns = [a_idx for a_idx, a in enumerate(ASSIGNMENTS) if a['subject'] == subj and a['class_id'] in cls_list]
    for a_idx in w_assigns:
        model.Add(sum(x[a_idx, 3, p] for p in PERIODS) == 1)
        model.Add(sum(x[a_idx, d, p] for d in [1, 2, 4, 5] for p in PERIODS) == 0)

# 軟約束：體育課避開第 5 節 (夏日防呆)
penalties = []
for a_idx, a in enumerate(ASSIGNMENTS):
    if '體育' in a['subject']:
        for d in DAYS:
            penalties.append(x[a_idx, d, 5] * 100)

if penalties:
    model.Minimize(sum(penalties))

solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 45.0
print('🚀 正在運算全校 24 班課表，請稍候...')
status = solver.Solve(model)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print('🎉 排課成功！')
    output_data = {}
    for c in CLASSES:
        output_data[c] = {'title': f'{c} 班級課表', 'homeroom': '導師', 'grid': {}}
        
    for a_idx, a in enumerate(ASSIGNMENTS):
        for d in DAYS:
            for p in PERIODS:
                if solver.Value(x[a_idx, d, p]) == 1:
                    c_id = a['class_id']
                    tag = ''
                    if any(a['subject'] == g['subject'] and c_id in g['classes'] for g in SYNC_GROUPS):
                        tag = '🔗 資源班抽離'
                    elif a['subject'] == '本土語':
                        tag = '閩/客二選一'
                    output_data[c_id]['grid'][f'{d}-{p}'] = {
                        'course': a['subject'],
                        'teacher': a['teacher'],
                        'tag': tag
                    }
                    
    # 寫入 json
    with open('timetable_data.json', 'w', encoding='utf-8') as f:
        json.dump(output_data, f, ensure_ascii=False, indent=2)
    print('📁 已產出 timetable_data.json！')
    files.download('timetable_data.json')
else:
    print('❌ 排課失敗，約束條件有衝突，請檢查設定！')